# 📚 LangChain Fundamentals — LLM Calls, Messages, Structured Output & Chains

> **Learning Objective:** Understand how to use LangChain to interact with LLMs (like Claude),
> structure inputs/outputs, and compose powerful pipelines using chains.

---

## 📦 Section 0 — Library Overview

Before we write any code, let's understand every library we'll use throughout this notebook.

| Library | Package | Purpose |
|---|---|---|
| `langchain_anthropic` | `langchain-anthropic` | LangChain integration for Anthropic's Claude models |
| `langchain_core` | `langchain-core` | Core abstractions: prompts, messages, output parsers, runnables |
| `langchain` | `langchain` | High-level utilities including `init_chat_model` factory |
| `pydantic` | `pydantic` | Data validation and schema definition using Python type hints |
| `typing` | built-in | Python's type hint system — `TypedDict`, `Optional`, `List`, etc. |
| `python-dotenv` | `python-dotenv` | Loads secret keys (API keys) from a `.env` file into environment variables |
| `os` | built-in | Access operating system environment variables (e.g., `os.environ`) |

### 🔑 Why `python-dotenv`?
Never hardcode API keys in your notebook. Store them in a `.env` file:
```
ANTHROPIC_API_KEY=sk-ant-xxxx
```
Then load them safely at runtime. This way secrets never appear in git history.

### 🔑 Why LangChain at all?
LangChain provides a **unified interface** across 50+ LLM providers. Switching from Claude to GPT-4
only requires changing one line. It also provides composable building blocks (chains, prompts,
parsers) that make complex LLM workflows easy to build and maintain.


---
## 🤖 Section 1 — Making Your First LLM Call

### Approach A: Direct `ChatAnthropic` initialization
Use this when you want explicit control over Anthropic-specific settings.


In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
# ChatAnthropic: LangChain wrapper around Anthropic's Claude API.
#   It exposes a standard .invoke() / .stream() interface regardless of provider.
from langchain_anthropic import ChatAnthropic

# load_dotenv: reads your .env file and pushes variables into os.environ
# find_dotenv: walks up the directory tree to locate the nearest .env file
#   so you don't need to hardcode the path (works from any working directory)
from dotenv import load_dotenv, find_dotenv

import os

# ─── Load secrets ─────────────────────────────────────────────────────────────
# This must run BEFORE any LLM initialization, because ChatAnthropic reads
# ANTHROPIC_API_KEY from os.environ at construction time.
load_dotenv(find_dotenv())

# ─── Model Initialization ─────────────────────────────────────────────────────
# model      : which Claude variant to use (haiku = fastest & cheapest)
# temperature: 0 = deterministic/factual responses
#              1 = creative/varied responses
model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

# ─── Invoke ───────────────────────────────────────────────────────────────────
# .invoke() sends a single request and waits for the full response.
# It accepts a plain string (auto-wrapped as a HumanMessage internally).
# Returns an AIMessage object — use .content to get the text.
response = model.invoke("Hello! What is the capital of UAE?")
print(response.content)


### Approach B: `init_chat_model` — Provider-Agnostic Factory

`init_chat_model` is LangChain's recommended modern approach. It automatically
detects the provider from the model name, making it easy to swap models later.


In [ ]:
# init_chat_model: smart factory that infers the provider from the model string.
#   "claude-haiku-4-5"  → uses langchain_anthropic under the hood
#   "gpt-4o"            → would use langchain_openai under the hood
# No provider import needed — langchain does the routing automatically.
from langchain.chat_models import init_chat_model

llm_claude = init_chat_model(model="claude-haiku-4-5")

# .invoke() with a plain string → treated as a HumanMessage
# .content extracts just the text from the returned AIMessage object
response = llm_claude.invoke("What is the probability of heads in a fair coin flip?").content
print(response)


### 💡 Concept: The `AIMessage` Return Object

Every `.invoke()` call returns an `AIMessage`. Key attributes:

| Attribute | Description |
|---|---|
| `.content` | The text response (what you usually want) |
| `.response_metadata` | Token usage, model name, stop reason |
| `.id` | Unique message ID |

```python
response = llm_claude.invoke("hello")
print(response.content)            # "Hello! How can I help you?"
print(response.response_metadata)  # {'model': 'claude-haiku-4-5', 'usage': {...}}
```


---
## 💬 Section 2 — Messages: System, Human, AI

LLMs are chat-based. A conversation is a **list of typed messages**:

| Message Type | Role | Purpose |
|---|---|---|
| `SystemMessage` | `system` | Sets the AI's persona/behavior for the entire conversation |
| `HumanMessage` | `user` | Represents the user's input |
| `AIMessage` | `assistant` | Represents a previous AI response (for multi-turn history) |

Passing a list of messages lets you control persona and simulate conversation history.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# ─── Multi-message conversation ───────────────────────────────────────────────
# SystemMessage: sets the AI persona for the whole session.
#   The model will respond "in character" for all following messages.
# HumanMessage: what the user is saying / asking.
my_messages = [
    SystemMessage(content="You are a character from Shakespeare's plays — speak in Early Modern English."),
    HumanMessage(content="How do I tell my girlfriend she is the prettiest girl in the whole world?")
]

# When you pass a list, LangChain sends it as a proper chat conversation.
response = llm_claude.invoke(my_messages)
print(response.content)


### 💡 Concept: Simulating Multi-Turn Conversation with `AIMessage`

You can inject previous AI responses to give the model memory of the conversation:
```python
history = [
    SystemMessage(content="You are a helpful Python tutor."),
    HumanMessage(content="What is a list?"),
    AIMessage(content="A list is an ordered, mutable collection of items."),
    HumanMessage(content="Give me an example."),   # ← model now has context
]
llm_claude.invoke(history).content
```
This is how chatbot memory is simulated — LangChain's `ConversationBufferMemory`
does exactly this automatically.


---
## 📝 Section 3 — Prompt Templates

Prompt Templates let you define **reusable prompt structures** with placeholders.
Instead of hardcoding prompts, you define them once and fill variables at runtime.

LangChain provides three main template types:

| Template | Use Case |
|---|---|
| `PromptTemplate` | Single string with `{variable}` placeholders |
| `ChatPromptTemplate` | Multi-role chat with system + user messages |
| `HumanMessagePromptTemplate` | Template for a single human message role |


In [ ]:
from langchain_core.prompts import PromptTemplate

# ─── PromptTemplate ───────────────────────────────────────────────────────────
# .from_template(): creates a template from a string with {variable} placeholders.
# Variables inside {} are filled at invoke time.
user_input = input("Enter a topic for a fun joke: ")
dynamic_prompt = PromptTemplate.from_template("Write a joke about {topic}")

# .invoke({variable: value}) fills the placeholders → returns a StringPromptValue
# This is a prepared prompt — not yet sent to the LLM.
ready_prompt = dynamic_prompt.invoke({"topic": user_input})

# Now pass the filled prompt to the LLM
response = llm_claude.invoke(ready_prompt).content
print(response)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# ─── ChatPromptTemplate ───────────────────────────────────────────────────────
# .from_messages(): accepts a list of (role, template_string) tuples.
# Both system and human messages can have {variable} placeholders.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an {tone} boss"),     # ← {tone} is dynamic
    ("user", "Requesting a leave on the {dates}")  # ← {dates} is dynamic
])

user_input = input("Which date do you want to apply for leave? ")
boss_tone = input("How should the boss react? (e.g. 'angry', 'understanding', 'formal'): ")

# .invoke() fills all placeholders → returns a ChatPromptValue (list of messages)
ready_prompt = prompt_template.invoke({"dates": user_input, "tone": boss_tone})

# Pass .messages (the list) to the LLM
response = llm_claude.invoke(ready_prompt.messages).content
print(response)


### 💡 Concept: `ChatPromptTemplate.from_messages` vs `PromptTemplate.from_template`

| | `PromptTemplate` | `ChatPromptTemplate` |
|---|---|---|
| Output | Single string | List of role-tagged messages |
| Best for | Simple completions | Chat models (system + user personas) |
| Variables | `{var}` in template | `{var}` in any role's template |

**When to use which?**
- Most modern models → use `ChatPromptTemplate` (it maps to the chat API format)
- Simple text templates / non-chat models → use `PromptTemplate`


---
## 🏗️ Section 4 — Structured Output with Pydantic Models

By default, LLMs return free-form text. **Structured output** forces the model
to return data matching a schema — like getting JSON with guaranteed fields.

### Why structured output?
- Downstream code can reliably access `result.setup` instead of parsing text
- Validation is automatic — wrong types are caught immediately
- Great for APIs, databases, and pipelines that need typed data


In [ ]:
from pydantic import BaseModel, Field

# ─── Define the Schema ────────────────────────────────────────────────────────
# BaseModel: Pydantic's base class — provides automatic validation
# Field(): adds metadata to each field:
#   description: tells the LLM what this field should contain
#   This description is used by LangChain to instruct the model.
class JokeSchema(BaseModel):
    setup: str = Field(description="The setup/premise of the joke")
    punchline: str = Field(description="The punchline that makes it funny")

# ─── Manual validation test ───────────────────────────────────────────────────
# Pydantic validates on construction — wrong types raise ValidationError
test_obj = JokeSchema(**{"setup": "Why did the programmer quit?", "punchline": "Because they didn't get arrays!"})
print("✅ Valid object:", test_obj)


In [ ]:
# ─── Bind schema to LLM ───────────────────────────────────────────────────────
# .with_structured_output(Schema): tells LangChain to:
#   1. Instruct the model to return JSON matching the schema
#   2. Automatically parse and validate the response into the Schema object
# Returns a new runnable — invoke it just like the base LLM.
llm_structured = llm_claude.with_structured_output(JokeSchema)

# The LLM now returns a JokeSchema instance, not raw text
result = llm_structured.invoke("Tell me a bar joke")

# Access fields directly — fully typed!
print("Setup:    ", result.setup)
print("Punchline:", result.punchline)


### 💡 Concept: Pydantic Field Validators

You can add validation logic beyond type checking:
```python
from pydantic import BaseModel, Field, field_validator

class StrictJoke(BaseModel):
    setup: str = Field(description="Setup for the joke", min_length=10)
    punchline: str = Field(description="Punchline for the joke", min_length=5)
    rating: int = Field(description="Joke rating from 1 to 5")

    @field_validator("rating")
    @classmethod
    def rating_must_be_valid(cls, v):
        if not 1 <= v <= 5:
            raise ValueError("Rating must be between 1 and 5")
        return v
```


---
## 📋 Section 5 — Structured Output with TypedDict

`TypedDict` is Python's built-in alternative to Pydantic for structured output.

| | `Pydantic BaseModel` | `TypedDict` |
|---|---|---|
| Validation | ✅ Full runtime validation | ❌ Type hints only (no runtime check) |
| Access | `result.setup` (attribute) | `result["setup"]` (dict key) |
| Extra dependencies | Requires pydantic | Built into Python |
| Best for | Production, APIs, strict validation | Lightweight/quick prototyping |


In [ ]:
from typing import TypedDict

# ─── TypedDict schema ─────────────────────────────────────────────────────────
# TypedDict: defines the expected keys and their types.
# No runtime validation — it's purely for type hints and IDE support.
class JokeTypedDict(TypedDict):
    setup: str      # key "setup" must be a string
    punchline: str  # key "punchline" must be a string

# ─── Bind to LLM ──────────────────────────────────────────────────────────────
# Works exactly like Pydantic — LangChain handles both transparently.
llm_typed = llm_claude.with_structured_output(JokeTypedDict)

result = llm_typed.invoke("Tell me a joke about Shah Rukh Khan")

# TypedDict returns a plain Python dict — access with square brackets
print("Setup:    ", result["setup"])
print("Punchline:", result["punchline"])


---
## ⛓️ Section 6 — Chains: Sequential Pipelines (LCEL)

**LangChain Expression Language (LCEL)** uses the `|` pipe operator to compose steps.
Each step's output becomes the next step's input — like Unix pipes.

### The 3 core building blocks of a chain:
1. **Prompt Template** → formats user input into a structured prompt
2. **LLM** → generates a response
3. **Output Parser** → converts the raw AIMessage into a usable format (e.g., plain string)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

# ─── Step 1: Prompt Template ──────────────────────────────────────────────────
# Defines the conversation structure. {input} will be filled at invoke time.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful and concise assistant."),
    ("human", "{input}")
])

# ─── Step 2: LLM ──────────────────────────────────────────────────────────────
llm_claude2 = init_chat_model(model="claude-haiku-4-5")

# ─── Step 3: Output Parser ────────────────────────────────────────────────────
# StrOutputParser: extracts the plain text string from an AIMessage object.
# Without it, the chain returns an AIMessage — with it, you get a clean string.
str_parser = StrOutputParser()


### Manual Step-by-Step Invocation (for understanding the data flow)

In [ ]:
# ─── Manual pipeline (step by step) ──────────────────────────────────────────
# This is what the | pipe does internally — useful to understand the data flow.

# Step 1: Template fills the {input} placeholder → returns ChatPromptValue
template_output = prompt_template.invoke({"input": "What is the capital of India?"})
print("After prompt template:", type(template_output))

# Step 2: LLM processes the formatted messages → returns AIMessage
llm_output = llm_claude2.invoke(template_output)
print("After LLM:", type(llm_output))

# Step 3: Parser extracts the string content → returns plain str
final_result = str_parser.invoke(llm_output)
print("After parser:", type(final_result))
print("\nFinal answer:", final_result)


### LCEL Pipe Syntax (the clean way)

In [ ]:
# ─── LCEL chain with | operator ───────────────────────────────────────────────
# The | operator connects runnables into a pipeline.
# Data flows left to right: prompt → LLM → parser
# This is equivalent to the manual steps above, but composable and reusable.
chain = prompt_template | llm_claude2 | str_parser

# .invoke() triggers the full pipeline end-to-end
result = chain.invoke({"input": "What is the capital of India?"})
print(result)


In [ ]:
# ─── Alternative: RunnableSequence (explicit version of |) ───────────────────
# RunnableSequence is what | creates under the hood.
# Use this when you need to build chains programmatically (e.g., from a list).
from langchain_core.runnables import RunnableSequence

chain2 = RunnableSequence(prompt_template, llm_claude2, str_parser)
result = chain2.invoke({"input": "What is the capital of the USA?"})
print(result)


---
## ⛓️ Section 7 — Multi-Step Chain: LLM → Transform → LLM

Chains can include **custom transformation functions** between LLM calls using `RunnableLambda`.
This lets you reshape data between steps.

**Pipeline:** `User Input → LLM (answer) → dict wrapper → LLM (LinkedIn post)`


In [ ]:
from langchain_core.runnables import RunnableLambda

# ─── Custom transformation step ───────────────────────────────────────────────
# RunnableLambda: wraps any Python function as a LangChain Runnable.
# This makes it compatible with the | pipe operator.
# Use it whenever you need to reshape/transform data between chain steps.
def dictionary_maker(text: str) -> dict:
    """Wraps a plain string in a dict with key 'text'.
    Needed because the next prompt template expects {text} as a dict key.
    """
    return {"text": text}

# Wrap the function so it's pipe-compatible
dictionary_maker_runnable = RunnableLambda(dictionary_maker)

# ─── Second prompt for social media post ──────────────────────────────────────
# This prompt receives {text} — the output from the first LLM call
prompt_post = ChatPromptTemplate.from_messages([
    ("system", "You are a social media post generator."),
    ("human", "Create a LinkedIn post for the following text: {text}")
])

llm_claude3 = init_chat_model(model="claude-haiku-4-5", temperature=0)
str_parser = StrOutputParser()

# ─── Full 2-LLM chain ─────────────────────────────────────────────────────────
# Flow: question → LLM1 (answers) → string → dict → LLM2 (makes LinkedIn post)
chain = (
    prompt_template          # formats user question
    | llm_claude2            # answers the question → AIMessage
    | str_parser             # extracts text → str
    | dictionary_maker_runnable  # {"text": <answer>} → dict
    | prompt_post            # formats into LinkedIn post prompt
    | llm_claude3            # generates the post → AIMessage
    | str_parser             # extracts text → str
)

result = chain.invoke({"input": "What is the capital of UAE?"})
print(result)


---
## 🔀 Section 8 — Parallel Chains with `RunnableParallel`

`RunnableParallel` runs multiple chains **simultaneously** on the same input.
Results are returned as a dict with your named keys.

**Use case:** Generate LinkedIn + Instagram posts at the same time from one answer.

```
                    ┌──────────────────┐ → LinkedIn Post
User Input → LLM → ├  RunnableParallel |
                    └──────────────────┘ → Instagram Post
```


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

# ─── Branch 1: LinkedIn chain ─────────────────────────────────────────────────
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator."),
    ("human", "Create a professional LinkedIn post for: {text}")
])
llm_linkedin = init_chat_model(model="claude-haiku-4-5", temperature=0)
str_parser = StrOutputParser()

# This is a complete sub-chain (will run as one branch of the parallel)
chain_linkedin = linkedin_prompt | llm_linkedin | str_parser


In [ ]:
# ─── Branch 2: Instagram chain (as a RunnableLambda) ─────────────────────────
# This demonstrates wrapping a full chain inside a function.
# Useful when the branch needs preprocessing of its input dict.
def insta_chain_fn(text: dict) -> str:
    """Receives a dict with 'text' key, generates an Instagram post."""
    topic = text["text"]  # extract the string from the dict

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator. Use emojis and hashtags."),
        ("human", "Create an Instagram post for: {text}")
    ])
    llm_insta = init_chat_model(model="claude-haiku-4-5", temperature=0)
    chain_insta = insta_prompt | llm_insta | StrOutputParser()

    return chain_insta.invoke({"text": topic})

# Wrap the function so it's pipe-compatible
insta_chain_runnable = RunnableLambda(insta_chain_fn)


In [ ]:
# ─── Orchestration: Run both branches in parallel ────────────────────────────
# RunnableParallel(branches={key: chain, ...}):
#   - Sends the same input to ALL branches simultaneously
#   - Collects results into a dict: {"branches": {"linkedin": ..., "instagram": ...}}
final_chain = (
    prompt_template           # Step 1: format user question
    | llm_claude3             # Step 2: answer the question
    | str_parser              # Step 3: extract text
    | dictionary_maker_runnable  # Step 4: wrap in {"text": ...} dict
    | RunnableParallel(
        branches={
            "linkedin": chain_linkedin,       # Branch A runs in parallel
            "instagram": insta_chain_runnable # Branch B runs in parallel
        }
    )
)

raw_result = final_chain.invoke({"input": "Chak de India"})
print(raw_result)


---
## ✨ Section 9 — Post-Processing: Beautifying Parallel Output

After `RunnableParallel`, the result is a nested dict. Add a final cleanup step
to reshape it into a cleaner, more user-friendly format.


In [ ]:
# ─── Beautify function ────────────────────────────────────────────────────────
# Extracts and flattens the nested parallel output into a simple dict.
# RunnableParallel returns: {"branches": {"linkedin": "...", "instagram": "..."}}
# We flatten it to:         {"linkedin": "...", "instagram": "..."}
def beautify(final_response: dict) -> dict:
    """Flatten the RunnableParallel nested output for clean downstream use."""
    linkedin_response = final_response["branches"]["linkedin"]
    instagram_response = final_response["branches"]["instagram"]
    return {
        "linkedin": linkedin_response,
        "instagram": instagram_response
    }

beautify_runnable = RunnableLambda(beautify)

# ─── Extend the chain with the cleanup step ───────────────────────────────────
beautify_chain = final_chain | beautify_runnable

result = beautify_chain.invoke({"input": "Chak de India"})
print("─── LinkedIn Post ───")
print(result["linkedin"])
print("\n─── Instagram Post ───")
print(result["instagram"])


---
## 🎁 Section 10 — Bonus: Important LangChain Concepts Not Covered Above

These are common patterns you'll encounter in real LangChain projects.


### 10A — Streaming Responses

Instead of waiting for the full response, stream tokens as they arrive.
Useful for chat UIs and long responses.


In [ ]:
from langchain.chat_models import init_chat_model

llm_stream = init_chat_model(model="claude-haiku-4-5")

# .stream() yields chunks of the response as they're generated
# Each chunk is an AIMessageChunk with a .content attribute
print("Streaming response:")
for chunk in llm_stream.stream("Tell me a short story about a robot in Dubai."):
    print(chunk.content, end="", flush=True)  # print without newlines = smooth stream
print()  # final newline


### 10B — Output Parsers: JSON & List

Beyond `StrOutputParser`, LangChain has parsers for common output formats.


In [ ]:
from langchain_core.output_parsers import JsonOutputParser, CommaSeparatedListOutputParser

# ─── CommaSeparatedListOutputParser ───────────────────────────────────────────
# Splits a comma-separated response into a Python list automatically.
list_parser = CommaSeparatedListOutputParser()
list_chain = (
    ChatPromptTemplate.from_template("List 5 {topic}, separated by commas.")
    | init_chat_model(model="claude-haiku-4-5")
    | list_parser
)
topics = list_chain.invoke({"topic": "data engineering tools"})
print("List output:", topics)
print("Type:", type(topics))  # → list

# ─── JsonOutputParser ─────────────────────────────────────────────────────────
# Parses JSON from the LLM response into a Python dict.
json_parser = JsonOutputParser()
json_chain = (
    ChatPromptTemplate.from_template(
        "Return a JSON object with keys 'city' and 'country' for: {location}"
    )
    | init_chat_model(model="claude-haiku-4-5")
    | json_parser
)
location_data = json_chain.invoke({"location": "Dubai"})
print("\nJSON output:", location_data)
print("City:", location_data["city"])


### 10C — RunnablePassthrough: Preserve Input Through a Chain

`RunnablePassthrough` passes the input unchanged to the next step.
Essential in RAG (Retrieval Augmented Generation) and multi-variable chains.


In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# ─── Preserve original input alongside LLM output ────────────────────────────
# Use case: you want both the original question AND the LLM answer downstream.
# RunnablePassthrough() passes input unchanged as one branch of the parallel.
preserve_chain = RunnableParallel({
    "original_question": RunnablePassthrough(),  # passes input dict as-is
    "answer": (
        ChatPromptTemplate.from_template("Answer briefly: {input}")
        | init_chat_model(model="claude-haiku-4-5")
        | StrOutputParser()
    )
})

result = preserve_chain.invoke({"input": "What is LangChain?"})
print("Question:", result["original_question"])
print("Answer:", result["answer"])


### 10D — Batch Processing

Run the same chain on multiple inputs efficiently using `.batch()`.
LangChain handles concurrency automatically.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

batch_chain = (
    ChatPromptTemplate.from_template("What is the capital of {country}?")
    | init_chat_model(model="claude-haiku-4-5")
    | StrOutputParser()
)

# .batch() runs the chain on ALL inputs, potentially in parallel
# More efficient than calling .invoke() in a loop
countries = [
    {"country": "UAE"},
    {"country": "India"},
    {"country": "USA"},
    {"country": "Japan"},
]

results = batch_chain.batch(countries)

for country, result in zip(countries, results):
    print(f"{country['country']}: {result.strip()}")


### 10E — Chain Introspection

LangChain chains are inspectable objects. You can examine their structure.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

simple_chain = (
    ChatPromptTemplate.from_template("Answer: {question}")
    | init_chat_model(model="claude-haiku-4-5")
    | StrOutputParser()
)

# .get_graph().print_ascii() — visualize the chain as a DAG
print("Chain structure:")
simple_chain.get_graph().print_ascii()

# .input_schema.schema() — see what inputs the chain expects
print("\nExpected input schema:")
print(simple_chain.input_schema.schema())


---
## 🗺️ LangChain Concept Map — Quick Reference

```
LangChain Building Blocks
│
├── 🧠 Models
│   ├── ChatAnthropic          → Direct Anthropic integration
│   └── init_chat_model()      → Provider-agnostic factory
│
├── 💬 Messages
│   ├── SystemMessage          → AI persona/behavior
│   ├── HumanMessage           → User input
│   └── AIMessage              → AI response (for history)
│
├── 📝 Prompt Templates
│   ├── PromptTemplate         → Single string with {vars}
│   └── ChatPromptTemplate     → Multi-role chat format
│
├── 🏗️ Structured Output
│   ├── Pydantic BaseModel     → Validated, attribute access
│   └── TypedDict              → Lightweight, dict access
│
├── 🔄 Output Parsers
│   ├── StrOutputParser        → AIMessage → str
│   ├── JsonOutputParser       → str → dict
│   └── CommaSeparatedList...  → str → list
│
└── ⛓️ Runnables (LCEL)
    ├── |  pipe operator       → Sequential chain
    ├── RunnableSequence       → Explicit sequential chain
    ├── RunnableLambda         → Wrap any Python function
    ├── RunnableParallel       → Run branches simultaneously
    ├── RunnablePassthrough    → Pass input unchanged
    └── .batch()               → Process multiple inputs
```
